# 04 - Analyze Results

Post-training analysis notebook for MBPS results.

**Goals:**
- Load and compare experiment results
- Generate ablation comparison tables
- Visualize per-class PQ breakdown
- Analyze failure cases
- Compare with baselines (CUPS, U2Seg)

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

RESULTS_DIR = '../results/'

## 1. Load Results

In [ ]:
def load_results(results_dir):
    results = {}
    results_path = Path(results_dir)
    if not results_path.exists():
        print(f'Results directory not found: {results_dir}')
        print('Run training first, then come back to analyze.')
        return results
    for json_file in results_path.glob('*.json'):
        with open(json_file) as f:
            results[json_file.stem] = json.load(f)
    return results

results = load_results(RESULTS_DIR)
print(f'Loaded {len(results)} experiment results')
for name in sorted(results.keys()):
    print(f'  - {name}')

## 2. Ablation Comparison

In [ ]:
# Compare PQ across ablations
ablation_names = [
    'full_model', 'no_mamba', 'no_depth_cond',
    'no_bicms', 'no_consistency', 'oracle_stuff_things'
]

metrics = ['PQ', 'PQ_Th', 'PQ_St', 'SQ', 'RQ', 'mIoU']

print(f'{"Experiment":<25}', end='')
for m in metrics:
    print(f'{m:>8}', end='')
print()
print('-' * (25 + 8 * len(metrics)))

for name in ablation_names:
    if name in results:
        r = results[name]
        print(f'{name:<25}', end='')
        for m in metrics:
            val = r.get(m, r.get(m.lower(), 0.0))
            print(f'{val:>8.1f}', end='')
        print()
    else:
        print(f'{name:<25} (not found)')

## 3. Baseline Comparison

In [ ]:
# Published baselines
baselines = {
    'CUPS (CVPR 2025)': {'PQ': 27.8, 'note': 'requires stereo video'},
    'U2Seg': {'PQ': 18.4, 'note': 'general unsupervised'},
}

print('Baseline Comparison on Cityscapes val:')
print(f'{"Method":<30} {"PQ":>6} {"Note":<30}')
print('-' * 70)

for name, info in baselines.items():
    print(f'{name:<30} {info["PQ"]:>6.1f} {info.get("note", ""):<30}')

if 'full_model' in results:
    pq = results['full_model'].get('PQ', results['full_model'].get('pq', 0))
    print(f'{"MBPS (Ours)":<30} {pq:>6.1f} {"monocular depth only":<30}')

## 4. Training Curves

In [ ]:
log_path = os.path.join(RESULTS_DIR, 'training_log.json')
if os.path.exists(log_path):
    with open(log_path) as f:
        logs = json.load(f)
    
    epochs = [e['epoch'] for e in logs]
    losses = [e.get('loss', 0) for e in logs]
    pqs = [e.get('pq', 0) for e in logs]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    ax1.plot(epochs, losses, 'b-')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training Loss')
    ax1.axvline(x=20, color='gray', linestyle='--', alpha=0.5, label='Phase B')
    ax1.axvline(x=40, color='gray', linestyle=':', alpha=0.5, label='Phase C')
    ax1.legend()
    
    ax2.plot(epochs, pqs, 'r-')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('PQ')
    ax2.set_title('Panoptic Quality')
    ax2.axhline(y=27.8, color='green', linestyle='--', label='CUPS')
    ax2.legend()
    
    plt.tight_layout()
    plt.show()
else:
    print('Training log not found. Run training first.')

## 5. Per-Class Analysis

In [ ]:
if 'full_model' in results and 'pq_per_class' in results['full_model']:
    pq_per_class = np.array(results['full_model']['pq_per_class'])
    
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(range(len(pq_per_class)), pq_per_class, color='steelblue', alpha=0.8)
    ax.set_xlabel('Class ID')
    ax.set_ylabel('PQ')
    ax.set_title('Per-Class Panoptic Quality')
    ax.axhline(y=np.mean(pq_per_class[pq_per_class > 0]), 
               color='red', linestyle='--', label='Mean PQ')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('Per-class results not available yet.')

## Summary

This notebook provides post-training analysis for the MBPS model.
Run after completing training and ablation experiments.